# Task 2 Demo

This notebook trains the vision model on the Kaggle animal dataset, trains the text model on the synthetic labeled JSON dataset, and lets you verify your own text against your own uploaded image.

The notebook runs training and verification in the project Python process instead of importing the full pipeline into the notebook kernel. This avoids common VS Code Jupyter issues around `torch` and widget callbacks.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys
import uuid

import ipywidgets as widgets
from IPython.display import Image as DisplayImage, display

ROOT = Path.cwd()
PROJECT_ROOT_CANDIDATES = [
    ROOT,
    ROOT / 'task2_animals',
    ROOT.parent,
]
PROJECT_ROOT = next((path for path in PROJECT_ROOT_CANDIDATES if (path / 'src').exists()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Could not locate the task2_animals/src directory.')

PROJECT_PYTHON_CANDIDATES = [
    PROJECT_ROOT.parent / '.venv' / 'Scripts' / 'python.exe',
    Path(sys.executable),
]
PROJECT_PYTHON = next((path for path in PROJECT_PYTHON_CANDIDATES if path.exists()), None)
if PROJECT_PYTHON is None:
    raise FileNotFoundError('Could not locate a Python executable for the project environment.')

SRC_ROOT = PROJECT_ROOT / 'src'
DATA_DIR = PROJECT_ROOT / 'data'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

print('Notebook root:', PROJECT_ROOT)
print('Project python:', PROJECT_PYTHON)

def _run_project_json(user_code: str) -> dict:
    marker = '__TASK2_JSON__'
    wrapped_code = (
        'import json\n'
        'import sys\n'
        'from pathlib import Path\n'
        f"sys.path.insert(0, r'{SRC_ROOT}')\n"
        + user_code
        + f"\nprint('{marker}')\nprint(json.dumps(result))\n"
    )

    completed = subprocess.run(
        [str(PROJECT_PYTHON), '-c', wrapped_code],
        cwd=str(PROJECT_ROOT),
        capture_output=True,
        text=True,
    )

    if completed.returncode != 0:
        raise RuntimeError(
            'Project command failed.\n'
            f'STDOUT:\n{completed.stdout}\n'
            f'STDERR:\n{completed.stderr}'
        )

    if completed.stderr.strip():
        print(completed.stderr.strip())

    if marker not in completed.stdout:
        raise RuntimeError(f'JSON marker was not found in command output.\nSTDOUT:\n{completed.stdout}')

    _, payload = completed.stdout.split(marker, 1)
    return json.loads(payload.strip())

def _display_verification_result(result):
    print(f"Text animal: {result['text_animal']}")
    print(f"Predicted animal: {result['predicted_animal']}")
    print(f"Match: {result['is_match']}")
    print(f"Supported animal classes: {result['supported_animals_count']}")
    print('\nTop image predictions:')
    for item in result['image_result']['top_k']:
        print(f"- {item['label']}: {item['confidence']:.4f}")

    print('\nExtracted text entities:')
    if result['text_entities']:
        for entity in result['text_entities']:
            print(f"- {entity['text']} ({entity['label']}, score={entity['score']:.4f})")
    else:
        print('- none')


## 1. Train both models

This demo cell uses only 10 animal classes to reduce training time.
If you want the full training run, change `max_classes=10` to `max_classes=None`.

In [ ]:
training_result = _run_project_json(f"""
from pipeline.animal_verifier import AnimalVerifier

verifier = AnimalVerifier(data_dir=Path(r'{DATA_DIR}'))
result = verifier.fit(
    max_classes=10,
    text_samples_per_class=32,
    image_num_epochs=8,
    image_learning_rate=3e-4,
    text_num_epochs=5,
)
""")

image_training = training_result['image_training']
print('Supported animal classes:', image_training['num_classes'])

best_val_accuracy = image_training.get('best_val_accuracy')
if best_val_accuracy is not None:
    print('Best validation accuracy:', round(best_val_accuracy, 4))

test_metrics = image_training.get('test_metrics')
if test_metrics is not None:
    print('Image test metrics:', test_metrics)

training_result['prepared']

## 2. Type your text and upload your image

In [ ]:
for widget_name in ('task2_ui', 'text_input', 'image_upload'):
    widget = globals().get(widget_name)
    if widget is not None and hasattr(widget, 'close'):
        widget.close()

text_input = widgets.Textarea(
    value='I think this is a tiger.',
    placeholder='Type your text here',
    description='Text:',
    layout=widgets.Layout(width='800px', height='120px'),
)

image_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False,
    description='Upload image',
)

def _read_uploaded_file(upload_value):
    if isinstance(upload_value, dict):
        return next(iter(upload_value.values()))
    if isinstance(upload_value, (tuple, list)):
        return upload_value[0]
    raise TypeError(f'Unsupported upload payload type: {type(upload_value)}')

task2_ui = widgets.VBox([text_input, image_upload])
display(task2_ui)

## 3. Run verification

After you type the text and upload one image above, run this cell once.

In [ ]:
if not image_upload.value:
    raise ValueError('Please upload an image first.')

uploaded = _read_uploaded_file(image_upload.value)
image_bytes = bytes(uploaded['content'])
filename = uploaded['name']

uploads_dir = DATA_DIR / 'uploads'
uploads_dir.mkdir(parents=True, exist_ok=True)
image_path = uploads_dir / f"{uuid.uuid4().hex}_{Path(filename).name}"
image_path.write_bytes(image_bytes)

display(DisplayImage(data=image_bytes))

result = _run_project_json(f"""
from pipeline.animal_verifier import AnimalVerifier

verifier = AnimalVerifier(data_dir=Path(r'{DATA_DIR}'))
verifier.load()
result = verifier.verify_details(
    text={text_input.value!r},
    image_path=Path(r'{image_path}'),
    top_k=5,
)
""")

_display_verification_result(result)
result